# Phase 3b — LoRA SFT of Qwen3-4B-Thinking on distilled traces (Colab A100)

Same training logic as the prior version, with these changes:
- `MAX_SEQ_LEN` raised to 40960 (matches eval `MAX_MODEL_LEN`) so no distilled trace gets silently truncated. We *keep* every distilled response — they cost compute to generate.
- Section 5 now asserts the SFT data's system prompts exactly match the eval system prompts. Fails loud if anyone points this at the wrong file.
- Section 6 keeps the percentage-based loss-mask circuit-breaker AND adds a deterministic content check: the un-masked region of each batch sample must contain that sample's completion verbatim. This rules out the "masked the right *amount* but the wrong *half*" failure mode.
- Smoke logs every step instead of every 5 (4 smoke steps × logging_steps=5 would print nothing).

After training, run `eval_adapter.ipynb` for each checkpoint.

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/second_try'   # adjust if your folder is named differently
OUTPUT_DIR  = f'{PROJECT_DIR}/outputs'             # checkpoints land here — survives session resets
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('PROJECT_DIR:', PROJECT_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)

## 1. Install Unsloth + grader deps

Unsloth pulls its own compatible torch/transformers/trl pins. The one explicit pin: `antlr4-python3-runtime==4.11.1` because `sympy.parsing.latex` needs exactly that runtime; a mismatch silently breaks the grader.

After this cell, **Runtime → Restart**, then run from section 2 onward.

In [ ]:
!pip install -q unsloth 2>&1 | tail -3
!pip install -q sympy "antlr4-python3-runtime==4.11.1" 2>&1 | tail -1
print("Install done. NOW RESTART THE RUNTIME (Runtime > Restart), then run from section 2.")

## 2. Post-restart checks — run this first after the restart

In [ ]:
import os, sys
PROJECT_DIR = '/content/drive/MyDrive/second_try'
OUTPUT_DIR  = f'{PROJECT_DIR}/outputs'
sys.path.insert(0, PROJECT_DIR)

# Import order matters: unsloth FIRST (it patches transformers).
import unsloth
import torch, transformers, trl

print('unsloth     :', unsloth.__version__)
print('torch       :', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('transformers:', transformers.__version__)
print('trl         :', trl.__version__)
print('device      :', torch.cuda.get_device_name(0))
print('GPU free    :', round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), 'GB')

assert torch.cuda.is_available(), 'no GPU detected — pick A100 runtime'

# Grading-path sanity (antlr/sympy mismatch surfaces here)
from judger import Judger
j = Judger(strict_extract=False)
assert j.auto_judge(pred=r'\boxed{\frac{5}{8}}', gold=['5/8'], options=[[]]) is True, \
    'grading path broken — check sympy + antlr4-python3-runtime==4.11.1'
print('grading path: OK')

## 3. Config

`MAX_SEQ_LEN=40960` matches `eval_adapter.ipynb`'s `MAX_MODEL_LEN=40960`. Every distilled completion fits at this length (verified: max char-length of sys+user+completion is ~97k chars ≈ ~32k tokens; padding to 40960 leaves headroom for the chat-template wrapper).

In [ ]:
import json
from pathlib import Path

MODEL_ID    = 'unsloth/Qwen3-4B-Thinking-2507'
DATA_PATH   = f'{PROJECT_DIR}/sft_distilled.jsonl'
MAX_SEQ_LEN = 40960
SEED        = 151

# Training hyperparams
SMOKE       = True       # <- set False for the real 4-epoch run
EPOCHS      = 1 if SMOKE else 4
RANK        = 16
LR          = 2e-4
BATCH_SIZE  = 2
GRAD_ACCUM  = 4

# Resume from a prior checkpoint (set to e.g. f'{OUTPUT_DIR}/checkpoint-XXX')
RESUME_FROM = None

# Smoke uses a small subset (enough for >5 optimizer steps so loss can move)
SMOKE_N     = 64

# Logging: smoke logs every step so you actually see something in 4-8 steps;
# real run uses 5 to avoid spam over ~350 steps.
LOG_STEPS   = 1 if SMOKE else 5

print(f'SMOKE = {SMOKE} | epochs = {EPOCHS} | log_steps = {LOG_STEPS}')
print(f'data:   {DATA_PATH}')
print(f'output: {OUTPUT_DIR}')
print(f'resume_from: {RESUME_FROM}')
print(f'MAX_SEQ_LEN: {MAX_SEQ_LEN}')

## 4. Load model in bf16 + apply LoRA

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    dtype=torch.bfloat16,
)

model = FastModel.get_peft_model(
    model,
    r=RANK,
    lora_alpha=RANK * 2,
    lora_dropout=0,
    bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
print('model + LoRA loaded')

## 5. Build dataset — with hard prompt-match assertion

Asserts that every system prompt in the SFT data is exactly one of the two eval prompts. If you accidentally point `DATA_PATH` at the older 706-line file (which used different prompts), this fails loud before any training compute is wasted.

In [ ]:
from datasets import Dataset

# These MUST match eval_adapter.ipynb section 4 character-for-character.
# If you change one, change the other.
EXPECTED_SYS_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Give your final answer inside a single \\boxed{}. "
    "Use EXACT values: prefer fractions (\\frac{a}{b}) and symbolic forms "
    "(\\sqrt{}, \\pi, e) over decimals. If you must give a decimal, write at "
    "least 10 significant figures and do NOT round. "
    "If the problem has multiple sub-answers, put them all inside one \\boxed{}, "
    "comma-separated, in the order asked, e.g. \\boxed{41, 35, 16}. "
    "If a single sub-answer itself contains a comma (a point or tuple), wrap it "
    "in parentheses, e.g. \\boxed{(2, 3), 7}."
)
EXPECTED_SYS_MCQ = (
    "You are an expert mathematician. Read the problem and the answer choices, "
    "then select the single best answer. After your reasoning, output ONLY the "
    "letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}. "
    "The very last thing in your response must be that \\boxed{<letter>}."
)
ALLOWED_SYS = {EXPECTED_SYS_MATH, EXPECTED_SYS_MCQ}

rows = [json.loads(l) for l in open(DATA_PATH)]
print(f'loaded {len(rows)} records from {DATA_PATH}')

# HARD CHECK 1 — prompt match. The most catastrophic silent failure to guard against.
bad = [r['id'] for r in rows if r['messages'][0]['content'] not in ALLOWED_SYS]
if bad:
    sample_bad = rows[[r['id'] for r in rows].index(bad[0])]['messages'][0]['content']
    raise AssertionError(
        f'{len(bad)} records have system prompts that do NOT match the eval prompts.\n'
        f'First offending id={bad[0]}, prompt preview:\n  {sample_bad[:200]!r}\n'
        f'You probably loaded the wrong sft_distilled.jsonl. The 693-record version is correct.'
    )
print('prompt match: OK (all records use one of the two eval system prompts)')

# HARD CHECK 2 — completion well-formedness.
for r in rows:
    c = r['completion']
    assert c.lstrip().startswith('<think>'), f'id {r["id"]}: completion does not start with <think>'
    assert '</think>' in c, f'id {r["id"]}: completion has no </think>'
    assert '\\boxed{' in c, f'id {r["id"]}: completion has no \\boxed{{'
print('completion format: OK (all start with <think>, contain </think>, contain \\boxed{)')

if SMOKE:
    rows = rows[:SMOKE_N]
    print(f'SMOKE: using {len(rows)} examples')

def format_one(rec):
    full = rec['messages'] + [{'role': 'assistant', 'content': rec['completion']}]
    text = tokenizer.apply_chat_template(full, tokenize=False)
    return {'text': text, '_id': rec['id'], '_completion': rec['completion']}

formatted = [format_one(r) for r in rows]
train_ds = Dataset.from_list([{'text': f['text']} for f in formatted])
# Keep _id/_completion side-channel for the deterministic mask check in section 6.
id_to_completion = {f['_id']: f['_completion'] for f in formatted}
all_completions = [f['_completion'].strip() for f in formatted]

print(f'dataset built: {len(train_ds)} examples')

# Peek at the rendered chat template — eyeball that it looks like a Qwen3 chat
print('\n--- formatted example (first 400 chars) ---')
print(train_ds[0]['text'][:400])
print('\n--- ... (last 300 chars) ---')
print(train_ds[0]['text'][-300:])

# Length sanity
lens = [len(tokenizer(f['text']).input_ids) for f in formatted]
lens_sorted = sorted(lens)
p50 = lens_sorted[len(lens_sorted)//2]
p99 = lens_sorted[int(len(lens_sorted)*0.99)] if len(lens_sorted) >= 100 else lens_sorted[-1]
max_len = lens_sorted[-1]
n_over = sum(1 for l in lens if l > MAX_SEQ_LEN)
print(f'\ntoken length: p50={p50} | p99={p99} | max={max_len} | MAX_SEQ_LEN={MAX_SEQ_LEN}')
print(f'records exceeding MAX_SEQ_LEN (would be truncated): {n_over}')
if n_over > 0:
    print(f'  WARNING: {n_over} records will lose their tail (likely </think> or \\boxed{{}}).')
    print(f'  Either raise MAX_SEQ_LEN further or accept this loss explicitly.')

## 6. Trainer config + loss masking (with deterministic content check)

Two-layer check after `train_on_responses_only`:

1. **Percentage** (`10 < pct < 95`): rules out the catastrophic cases — mask covers 0% (template markers missed entirely → training on the whole sequence including the user's question) or 100% (response markers missed → training on nothing, loss stays at 0).

2. **Content** (deterministic): for each sample in the first batch, decode the un-masked tokens and verify that the resulting string contains a known completion verbatim. This catches "masked the right amount but the wrong half" — a failure the percentage cannot distinguish, where the model would learn to write *questions* instead of answers.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    optim='adamw_8bit',
    weight_decay=0.01,
    logging_steps=LOG_STEPS,
    save_strategy='epoch',
    save_total_limit=EPOCHS,
    bf16=True,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    seed=SEED,
    report_to='none',
    dataloader_num_workers=2,
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer,
                     train_dataset=train_ds, args=cfg)

# Compute loss ONLY on the assistant completion.
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

# Grab one batch for both circuit-breaker checks.
batch = next(iter(trainer.get_train_dataloader()))

# === Layer 1: percentage check ===
n_masked = (batch['labels'] == -100).sum().item()
n_total = batch['labels'].numel()
pct = 100 * n_masked / n_total
print(f'Mask coverage: {pct:.1f}% masked (system+user); training on {100-pct:.1f}% (completion).')
print(f'  Expected range for this dataset: ~15-40% (short prompts vs long thinking traces).')
assert 10 < pct < 95, f'Suspicious mask coverage ({pct:.1f}%) — template markers likely misaligned.'
print('  Layer-1 (percentage): OK')

# === Layer 2: deterministic content check ===
# For each sample, the un-masked region must contain a known completion verbatim.
# Membership over the full set of completions handles shuffle=True without needing
# to disable shuffling — math reasoning traces are essentially unique strings,
# so a substring match against the wrong completion is implausible.
N = batch['input_ids'].shape[0]
for i in range(N):
    mask = batch['labels'][i] != -100
    visible_ids = batch['input_ids'][i][mask]
    visible_text = tokenizer.decode(visible_ids, skip_special_tokens=False)
    
    matched = any(c in visible_text for c in all_completions)
    if not matched:
        print(f'\nSAMPLE {i} un-masked region (first 400 chars):')
        print(visible_text[:400])
        print(f'\nSAMPLE {i} un-masked region (last 200 chars):')
        print(visible_text[-200:])
        raise AssertionError(
            f'Sample {i}: un-masked region matches no known completion. '
            f'train_on_responses_only likely masked the wrong half — you would be '
            f'training on the question, not the answer. Fix the marker strings.'
        )
print(f'  Layer-2 (content): OK ({N}/{N} samples in batch — un-masked region contains the expected completion)')

# Show one un-masked region so you can eyeball it yourself.
mask0 = batch['labels'][0] != -100
vis0 = tokenizer.decode(batch['input_ids'][0][mask0], skip_special_tokens=False)
print('\n--- un-masked region of sample 0 (first 300 chars) ---')
print(vis0[:300])
print('\n--- ... (last 200 chars) ---')
print(vis0[-200:])
print('\nBoth halves above should look like assistant content (<think>...</think>\\boxed{...}<|im_end|>),')
print('NOT like a question.')

## 7. Train

In [ ]:
print(f'Training {EPOCHS} epoch(s), saving each to {OUTPUT_DIR}')
if RESUME_FROM:
    print(f'Resuming from: {RESUME_FROM}')
    trainer.train(resume_from_checkpoint=RESUME_FROM)
else:
    trainer.train()

print()
print('Training complete. Checkpoints saved to Drive:')
import os
for p in sorted(os.listdir(OUTPUT_DIR)):
    if p.startswith('checkpoint-'):
        print(f'  {OUTPUT_DIR}/{p}')

## 8. Smoke checklist

If this was a smoke run (SMOKE=True), confirm before flipping to the full run:

1. **Section 5 — prompt match assertion passed**. (If not, you're pointing at the wrong file.)
2. **Section 5 — `records exceeding MAX_SEQ_LEN` is 0**. (Otherwise some completions would be silently chopped.)
3. **Section 6 — `pct` is in the 15-40% range**. Outside that, even if the assertion passed, something is suspicious.
4. **Section 6 — both layer-1 and layer-2 checks passed**.
5. **Section 6 — eyeballing the printed un-masked region**: it should start with `<think>` (or right after the assistant role marker) and end with `\boxed{...}<|im_end|>`. If it instead shows the original problem statement, the mask is reversed.
6. **Section 7 — training loss decreased monotonically** over the 8 steps (`SMOKE_N=64` / batch 2 / grad-accum 4 = 8 optimizer steps with logging_steps=1, so you should see 8 loss prints).

All six green → set `SMOKE = False` in section 3 and re-run sections 3–7 for the real 4-epoch run.

## 9. Next steps

After the full run, eval each checkpoint against the held-out val set. Open `eval_adapter.ipynb` and run it once per checkpoint, changing `ADAPTER_PATH` each time.